# 04 Self Attention from Scratch

## Problem

self-attention 为什么能让每个 token 根据上下文重新组织自己的表示？Q、K、V 到底各自干什么，causal mask 又为什么是 decoder-only 模型的硬约束？

## Dependencies

建议先完成 `01`、`02`、`03`。至少要知道向量、矩阵、softmax、token embedding 和 shape 是什么。


## Goals

- 能用自己的话解释 Q、K、V 的分工
- 理解 attention score 是怎么来的，为什么要除以 `sqrt(d_k)`
- 理解为什么 attention 的输出不是 `Q`，也不是 `K`，而是对 `V` 的加权汇总
- 理解 causal mask 为什么能阻止模型偷看未来 token
- 能从零复述一个单头 self-attention 的完整前向过程

## Scope and Approach

这一节不会把 attention 压缩成一句 `softmax(QK^T)V` 就结束。我们会把它拆成一个真正可理解的过程：

1. 当前位置到底在“问”什么。
2. 所有历史位置到底在“提供”什么。
3. 分数是怎么变成权重的。
4. 为什么最后真正被汇总的是 `V`。


## self-attention 想解决什么问题

理解一句话里的某个 token，往往需要看别的 token。

比如：

- 看到“它”，你要回头找指代对象。
- 看到“因为”，你要找原因部分。
- 看到代码里的变量名，你要回头找它在哪里定义。

self-attention 的核心思想，就是让**每个位置自己决定：我要看谁、看多少、把看来的信息怎么混回自己。**

这和固定窗口不一样。固定窗口是“你只能看附近几个位置”；attention 是“你可以看所有允许看的位置，但看多少由内容相关性决定”。


## Q、K、V 到底是什么

Q、K、V 最容易被背成三个字母，最难的是知道它们为什么要分成三套表示。

可以先这样记：

- `Query`：我当前这个位置在找什么信息。
- `Key`：我这个位置能不能匹配到你的需求。
- `Value`：如果你决定关注我，你最终拿走的内容是什么。

所以 attention 的完整逻辑不是“把别人的内容抄过来”，而是：

1. 先用 `Q` 和 `K` 算匹配分数。
2. 再把分数变成权重。
3. 最后用权重对 `V` 做加权汇总。

这也是为什么注意力输出会依赖 `V`：`Q` 和 `K` 负责决定看谁，`V` 负责决定真正拿回什么内容。


In [ ]:
import numpy as np

# 固定随机种子，让输出稳定，便于读者反复观察同一个例子。
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

# 这里构造一个非常小的输入序列。
# x.shape = (seq_len, hidden_dim) = (4, 4)
# 可以把它理解成：4 个位置，每个位置当前有 4 维表示。
x = np.array([
    [1.0, 0.0, 0.5, 0.2],
    [0.8, 0.1, 0.3, 0.0],
    [0.1, 1.0, 0.4, 0.6],
    [0.0, 0.9, 0.2, 0.8],
])

# 三个投影矩阵分别把原始表示投到 Q、K、V 空间。
# 这里为了简单起见，输入维度和输出维度都设成 4。
W_q = np.random.randn(4, 4) * 0.2
W_k = np.random.randn(4, 4) * 0.2
W_v = np.random.randn(4, 4) * 0.2

# Q、K、V 的 shape 都会是 (seq_len, hidden_dim)
Q = x @ W_q
K = x @ W_k
V = x @ W_v

print('x.shape =', x.shape)
print('Q.shape =', Q.shape)
print('K.shape =', K.shape)
print('V.shape =', V.shape)


## 第一步：先算相关性分数

对于当前位置 `i`，我们拿它的 `q_i` 去和所有位置的 `k_j` 做点积。点积越大，表示 `i` 觉得 `j` 越相关。

如果把所有位置一起算，就得到一个 `scores` 矩阵：

- 行：当前是谁在看别人
- 列：当前在看谁

所以 `scores.shape = (seq_len, seq_len)`。

这一步还只是“打分”，不是最终输出。


In [ ]:
def softmax(logits):
    # 经典的数值稳定写法：先减去每一行最大值。
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

seq_len, d_k = Q.shape

# attention score 的核心计算：Q @ K.T
# Q.shape = (seq_len, d_k)
# K.T.shape = (d_k, seq_len)
# 所以 scores.shape = (seq_len, seq_len)
scores = Q @ K.T

# 为什么要除以 sqrt(d_k)？
# 因为维度越大，点积通常会越大，softmax 会更容易过尖。
# 缩放以后，数值范围更稳定。
scaled_scores = scores / np.sqrt(d_k)

print('raw scores =')
print(scores)
print('scaled scores =')
print(scaled_scores)


## 第二步：为什么要加 causal mask

如果这是 decoder-only 语言模型，那么第 `t` 个位置在预测时不能看 `t+1` 以后的内容。否则模型训练时就等于在偷看答案。

causal mask 的做法不是把未来位置删掉，而是：

- 未来位置对应的分数加一个极小的负数
- 这样经过 softmax 之后，它们的权重几乎就是 0

这一步是语言模型成立的关键约束之一。


In [ ]:
# 构造一个上三角 mask。
# 对角线以上的位置表示“未来”，它们不允许被当前 token 看到。
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -1e9
masked_scores = scaled_scores + causal_mask

print('causal_mask =')
print(causal_mask)
print('masked_scores =')
print(masked_scores)


## 第三步：分数怎么变成权重

现在每一行都有一组“我看谁更相关”的分数，但分数本身还不能直接拿来做加权。softmax 的作用就是把每一行分数变成：

- 全都大于 0
- 总和等于 1
- 可以解释成注意力权重

经过这一步之后，我们终于得到“每个位置看每个历史位置的程度”。


In [ ]:
weights = softmax(masked_scores)

print('attention weights =')
print(weights)
print('每一行权重和 =', weights.sum(axis=-1))


## 第四步：为什么最后汇总的是 V

这是 attention 最容易误解的一步。

很多人看到 `QK^T` 会以为 attention 的输出已经差不多出来了。其实不是。`QK^T` 只是在决定“看谁”。真正的输出来自：

- 拿到一行权重
- 用它对所有位置的 `V` 做加权平均

所以可以这么记：

- `Q` 决定我在找什么
- `K` 决定谁和我匹配
- `V` 决定我真正拿走什么内容


In [ ]:
# 最终输出来自 weights @ V
# weights.shape = (seq_len, seq_len)
# V.shape       = (seq_len, hidden_dim)
# output.shape  = (seq_len, hidden_dim)
output = weights @ V

print('output.shape =', output.shape)
print('attention output =')
print(output)


## Common mistakes

- 把 `Q @ K.T` 理解成在搬运内容。它其实只是在算相关性分数。
- 以为 attention weights 本身就是输出。真正的输出是 `weights @ V`。
- 忽略 causal mask，导致语言模型在训练时偷看未来 token。
- 只背 `softmax(QK^T)V`，但说不清每一项在系统里分别承担什么角色。

## Checkpoints

- 去掉 `causal_mask`，观察每一行会不会开始看未来位置。
- 把 `W_q`、`W_k`、`W_v` 改成不同随机种子，观察注意力矩阵怎么变化。
- 用自己的话解释：为什么 attention 的输出不是 `Q`，也不是 `K`，而是混合后的 `V`。
- 说明为什么 attention 的计算复杂度随着序列长度增长到 `O(n^2)`。

## Summary

单头 self-attention 已经足够展示 Transformer 的核心思想：每个位置都能动态决定该参考哪些上下文，而不是像固定窗口那样机械地看邻居。`Q` 决定“我在找什么”，`K` 决定“谁值得我看”，`V` 决定“我真正拿走什么内容”。

## Next Module

下一节进入 multi-head attention。你会看到为什么一个头通常不够，以及“多头”到底多了什么。
